In [6]:
from ragatouille import RAGPretrainedModel
RAG = RAGPretrainedModel.from_pretrained("colbert-ir/colbertv2.0")

artifact.metadata: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/405 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[Aug 02, 12:59:08] Loading segmented_maxsim_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...


W0802 12:59:09.521000 12556 site-packages\torch\utils\cpp_extension.py:480] Error checking compiler version for cl: [WinError 2] The system cannot find the file specified


CalledProcessError: Command '['where', 'cl']' returned non-zero exit status 1.

In [ ]:
import json
import re
import sys
from datetime import datetime
# import faiss
# import numpy as np
# import requests
from langchain_text_splitters import RecursiveCharacterTextSplitter
# import os
# os.environ["COLBERT_DISABLE_EXTENSIONS"] = "1"
# os.environ["COLBERT_LOAD_TORCH_EXTENSION_VERBOSE"] = "0"
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# from sentence_transformers import SentenceTransformer

all_text = json.loads(open("context_list.json", "r+").read())
query = "How many leave days do employees get?"
try:
    total_texts = ""
    chunk_list = []
    splitter = RecursiveCharacterTextSplitter(
        separators="\n",
        chunk_overlap=0
    )

    for page_num, context in enumerate(all_text[:]):
        page_txt = f"\n\n\t\tThe Above Text is from the page number {page_num + 1}.\n\n"
        splTxt = splitter.split_text(re.sub(r"\s+", " ", context))
        for txt in splTxt:
            # txt = txt + page_txt
            chunk_list.append(txt)
        total_texts += context + page_txt
    if False:
    # if len(total_texts) < 50000:
        p=0
        # return total_texts
    else:
        # RAG = getRag(logfile)
        # RAG = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        top_k = int(len(all_text) * 0.2)
        # # sentence transformation
        # embeddings = RAG.encode(all_text, convert_to_numpy=True)
        # index = faiss.IndexFlatL2(embeddings.shape[1])
        # index.add(embeddings)
        # q_emb = RAG.encode([query], convert_to_numpy=True)
        # D, I = index.search(q_emb, k=top_k)
        # total_texts = "\n".join([all_text[i] for i in I[0]])

        # # colbert
        RAG.index(
            collection=chunk_list,
            document_ids=[str(page + 1) for page, context in enumerate(chunk_list)],
            index_name="crux",
            overwrite_index=True,
            max_document_length=512,
            split_documents=True,
            use_faiss=True
        )
        results = RAG.search(query=query, k=top_k)
        print(results)
        total_texts = "\n".join([data["content"] for data in results])

except Exception as e:
    print(e)